# Sample Data Creation Script

## Imports

In [1]:
import numpy as np
import pandas as pd

from k_anonymization import datasets

## Data Creation

### Original Data

In [2]:
DATASET = datasets.LONDON_HOUSE_PRICE
org_df = DATASET.df[DATASET.df.year == 2002].reset_index(drop=True)
org_df.shape[0]

35309

### Unique-QIDs Data

In [3]:
freq_org_df = org_df.groupby(DATASET.qids, as_index=False).count()
org_unique_indices = freq_org_df[freq_org_df[DATASET.target] == 1].index

In [4]:
org_unique_df = pd.merge(
    org_df.reset_index(),
    freq_org_df[freq_org_df[DATASET.target] == 1][DATASET.qids],
    on=DATASET.qids,
    how="right",
)
org_unique_df

,index,uuid,year,month,district,street,is_new,price,price_class
0,10699,130617,2002,1,BARNET,ALEXANDRA GROVE,False,160000,<=200k
1,33052,152970,2002,1,BARNET,ALEXANDRA ROAD,False,92250,<=200k
2,17339,137257,2002,1,BARNET,ASHBOURNE CLOSE,False,80000,<=200k
3,23520,143438,2002,1,BARNET,ASHFIELD ROAD,False,120000,<=200k
4,11844,131762,2002,1,BARNET,ASHURST ROAD,False,152000,<=200k
...,...,...,...,...,...,...,...,...,...
14828,34984,154902,2002,12,WANDSWORTH,WEXFORD ROAD,False,320000,>200k
14829,30852,150770,2002,12,WANDSWORTH,WINDLESHAM GROVE,False,180000,<=200k
14830,10745,130663,2002,12,WANDSWORTH,WOODBOROUGH ROAD,False,435000,>200k
14831,26446,146364,2002,12,WANDSWORTH,WYE STREET,False,79000,<=200k


#### Unique-QIDs Data, Target = "$\tiny\leq$200K"

In [5]:
org_unique_df_A = org_unique_df[
    org_unique_df[DATASET.target] == org_unique_df.loc[0, DATASET.target]
]
org_unique_df_A

,index,uuid,year,month,district,street,is_new,price,price_class
0,10699,130617,2002,1,BARNET,ALEXANDRA GROVE,False,160000,<=200k
1,33052,152970,2002,1,BARNET,ALEXANDRA ROAD,False,92250,<=200k
2,17339,137257,2002,1,BARNET,ASHBOURNE CLOSE,False,80000,<=200k
3,23520,143438,2002,1,BARNET,ASHFIELD ROAD,False,120000,<=200k
4,11844,131762,2002,1,BARNET,ASHURST ROAD,False,152000,<=200k
...,...,...,...,...,...,...,...,...,...
14823,28786,148704,2002,12,WANDSWORTH,UPPER TOOTING ROAD,False,175000,<=200k
14824,24407,144325,2002,12,WANDSWORTH,VERMONT ROAD,False,162500,<=200k
14829,30852,150770,2002,12,WANDSWORTH,WINDLESHAM GROVE,False,180000,<=200k
14831,26446,146364,2002,12,WANDSWORTH,WYE STREET,False,79000,<=200k


#### Unique-QIDs Data, Target = ">200K"

In [6]:
org_unique_df_B = org_unique_df[
    org_unique_df[DATASET.target] != org_unique_df.loc[0, DATASET.target]
]
org_unique_df_B

,index,uuid,year,month,district,street,is_new,price,price_class
9,9094,129012,2002,1,BARNET,BRIAR CLOSE,True,235000,>200k
10,27589,147507,2002,1,BARNET,BRINSDALE ROAD,False,400000,>200k
19,17280,137198,2002,1,BARNET,FINCHLEY ROAD,False,237500,>200k
20,30174,150092,2002,1,BARNET,GLOUCESTER GARDENS,False,297000,>200k
26,22769,142687,2002,1,BARNET,HENDON AVENUE,False,307000,>200k
...,...,...,...,...,...,...,...,...,...
14825,19173,139091,2002,12,WANDSWORTH,WAKEHURST ROAD,True,275000,>200k
14826,5755,125673,2002,12,WANDSWORTH,WARRINER GARDENS,False,221000,>200k
14827,15591,135509,2002,12,WANDSWORTH,WAYNFLETE STREET,False,206000,>200k
14828,34984,154902,2002,12,WANDSWORTH,WEXFORD ROAD,False,320000,>200k


### * Sample Data

Use a seed for reproducibility

In [7]:
SEED = 202602

In [8]:
sample_A = org_unique_df_A.sample(n=1500, random_state=SEED)
sample_B = org_unique_df_B.sample(n=1500, random_state=SEED)
sample = pd.concat([sample_A, sample_B], axis=0).sample(frac=1, random_state=SEED)
sample

,index,uuid,year,month,district,street,is_new,price,price_class
13029,23157,143075,2002,11,ISLINGTON,CLERKENWELL ROAD,False,350000,>200k
8835,21884,141802,2002,8,CITY OF WESTMINSTER,PORCHESTER GATE,False,1100000,>200k
14579,4697,124615,2002,12,RICHMOND UPON THAMES,UPPER RICHMOND ROAD WEST,False,170000,<=200k
1512,11204,131122,2002,2,HARINGEY,HILLCREST,False,209950,>200k
4746,1622,121540,2002,5,CAMDEN,SANDWICH STREET,False,170000,<=200k
...,...,...,...,...,...,...,...,...,...
419,14057,133975,2002,1,HAMMERSMITH AND FULHAM,AVONMORE ROAD,False,245000,>200k
8963,22061,141979,2002,8,GREENWICH,HADDO STREET,False,120000,<=200k
5207,1297,121215,2002,5,HARINGEY,RINGSLADE ROAD,False,120000,<=200k
7388,5706,125624,2002,7,CITY OF WESTMINSTER,HALL PLACE,False,220000,>200k


### * Validation Data

In [9]:
unsampled_A = org_unique_df_A[~org_unique_df_A.index.isin(sample_A.index)]
unsampled_B = org_unique_df_B[~org_unique_df_B.index.isin(sample_B.index)]

In [10]:
predict_org_A = unsampled_A.sample(n=300, random_state=SEED)
predict_org_B = unsampled_B.sample(n=300, random_state=SEED)
predict_org = pd.concat([predict_org_A, predict_org_B], axis=0).sample(
    frac=1, random_state=SEED
)
predict_org

,index,uuid,year,month,district,street,is_new,price,price_class
14729,3042,122960,2002,12,WALTHAM FOREST,BULWER ROAD,False,166950,<=200k
1793,11532,131450,2002,2,LEWISHAM,MALPAS ROAD,False,111000,<=200k
9045,15237,135155,2002,8,HAMMERSMITH AND FULHAM,CHESILTON ROAD,False,224000,>200k
6659,24196,144114,2002,6,LAMBETH,ENGLEWOOD ROAD,False,239950,>200k
3417,29668,149586,2002,4,CAMDEN,ENDELL STREET,False,350000,>200k
...,...,...,...,...,...,...,...,...,...
3950,8821,128739,2002,4,ISLINGTON,ST JOHNS VILLAS,False,180000,<=200k
2977,3112,123030,2002,3,NEWHAM,JEDBURGH ROAD,False,140000,<=200k
7354,26634,146552,2002,7,CITY OF WESTMINSTER,CABBELL STREET,False,495000,>200k
14277,21146,141064,2002,12,ISLINGTON,NEW NORTH ROAD,False,190000,<=200k


## Save Data

In [11]:
import os

BASE_PATH = f"./results/{DATASET.name.replace(" ", "_")}"
os.makedirs(f"{BASE_PATH}/sample", exist_ok=True)

In [12]:
sample.to_csv(f"{BASE_PATH}/sample/data.csv", index=False)

In [13]:
predict_org[org_df.columns].to_csv(f"{BASE_PATH}/predict_org.csv", index=False)